# 03 · Pair features

For every candidate pair (train and test) → `artifacts/features/{split}/{country}/part-*.parquet`.

**Retrieval** — `cos_<channel>` and `rank_<channel>` for all channels.

**Query context** — number of candidates, gap to the S1's best cosine and rank, per channel.

**Target competition** — how many S1 entities retrieved this record, its rank among them, gap/margin to the best competing S1. (Exclusivity holds perfectly in training data.)

**Name** — RapidFuzz ratio / token-sort / token-set / partial / Jaro-Winkler on `name_core`; ratio on `name_norm`, `name_alt` (DBA), phonetic skeleton and space-free core (`brighttavern` vs `bright tavern`); token Jaccard, first-token match, lengths, `name_freq` of both sides.

**Address** — ratio / token-set / token-sort on `addr_norm`, token Jaccard, house-number overlap / conflict / subset, missing-address flags.

**Record** — target is S3, target had non-Latin script.

Country is deliberately *not* a feature (France is unseen). All scorers run via `rapidfuzz.process.cpdist` (C++, multithreaded).

In [ ]:
# --- Setup: make src/ importable, load run settings -------------------------
import os, sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "entity_forge").is_dir())
sys.path.insert(0, str(ROOT / "src"))

# Override settings here or with EF_* environment variables before starting Jupyter.
# os.environ["EF_DEV_MODE"] = "1"     # small consistent slice: end-to-end smoke test on a laptop
# os.environ["EF_N_THREADS"] = "32"

import polars as pl
from entity_forge import stages
from entity_forge.settings import Settings

pl.Config.set_tbl_rows(30); pl.Config.set_fmt_str_lengths(80); pl.Config.set_tbl_width_chars(220)
stages.setup_logging()
S = Settings.from_env()
print(f"root={S.root}\nwork_dir={S.work_dir}\ndev_mode={S.dev_mode} threads={S.n_threads}")

In [ ]:
train_stats = stages.run_features(S, "train")
train_stats

In [ ]:
test_stats = stages.run_features(S, "test")
test_stats

## Sanity check

In [ ]:
ckey = stages.countries(S, "train")[0]
part = pl.read_parquet(stages.feature_parts(S, "train", ckey)[0])
print(part.shape, "positive rate:", round(part["label"].mean(), 4))
part.group_by("label").agg(pl.col("cos_both", "n_tset", "a_tset", "num_jacc", "t_margin_cos_both").mean())